# Distribution shift and test-time transformations

**MedLIME Learning Track — Notebook 08/12** · Curriculum weeks 25–28 · Suggested time: 60–100 minutes

**Before starting:** Notebooks 03–07.

**Learning goal:** Diagnose scanner-like appearance changes and understand input transformation families.

> **Important:** All images below are synthetic visual teaching phantoms, **not real medical images**. The educational implementations are **not** the original MedLIME implementation, are not evaluated for clinical use, and do not establish diagnostic or explanation reliability.


In [ ]:
from pathlib import Path
import sys
here = Path.cwd()
for candidate in (here, here.parent, here.parent.parent):
    if (candidate / 'medlime_tutorials.py').exists():
        sys.path.insert(0, str(candidate.resolve())); break
else:
    raise FileNotFoundError('Open Jupyter from the MedLIME folder or its notebooks/ subfolder')
import numpy as np
import matplotlib.pyplot as plt
from medlime_tutorials import *
np.set_printoptions(precision=3, suppress=True)


## A simple distribution shift
A model trained on one acquisition protocol may receive darker or lower-contrast images from another protocol. We simulate that situation numerically without pretending the effect captures real scanner physics.

**Terms:** source distribution = images used to establish a model; target distribution = images encountered after deployment. A transform \(T_\phi(x)=\mathrm{clip}(a x+b)\) changes brightness and contrast.

In [ ]:
x,gt=synthetic_image(seed=36,lesion=True)
dark=x*.58
bright=np.clip(x*1.15+.06,0,1)
g=ToyBlackBox()
show_images([x,dark,bright],['source-like','target: darker','target: brighter'])
plt.show()
for name,z in zip(['source','dark','bright'],[x,dark,bright]):
 print(name,'mean',round(float(z.mean()),3),'probability',round(float(g.predict_proba(z)[0]),5))

## Check a population, not only a selected image
Applying a transformation that helps one positive example could harm other patients or healthy images. Measure behavior across both classes.

In [ ]:
images,labels,_=dataset(80,seed=82)
scores_source=g.predict_proba(images)
scores_dark=g.predict_proba(images*.58)
print('Fraction called positive at threshold .5:')
for name,s in [('source',scores_source),('dark',scores_dark)]:
 print(name,'true positives',round(np.mean(s[labels==1]>.5),3),
       'false positives',round(np.mean(s[labels==0]>.5),3))
plt.hist(scores_source,bins=12,alpha=.5,label='source');plt.hist(scores_dark,bins=12,alpha=.5,label='dark')
plt.xlabel('Predicted positive probability');plt.ylabel('Count');plt.legend();plt.show()

## Why optimization enters the picture
Instead of selecting brightness by eye, we could choose trainable \(a,b\) using an objective. But which supervision signals are legitimately available at test time? If you use the true label of the query, that creates a supervised setting that differs from label-free deployment.

In [ ]:
transforms=[AffineAdapter(scale=s,bias=b) for s,b in [(1,0),(1.2,.0),(1.0,.12),(1.2,.12)]]
for t in transforms:
 print(f'scale={t.scale:.1f}, bias={t.bias:.2f}: p={g.predict_proba(t.apply(dark))[0]:.4f}')

## What you should now understand
- Brightness and contrast shifts can change model behavior even if pathology is unchanged.
- A transformation family has parameters that can be optimized.
- Test-time supervision availability must be stated, not assumed.

## Try it yourself (do not just read the code)
1. Plot score changes for darker shifts at factors 0.9, 0.7, 0.5.
2. Find a global brightness change that helps positives but produces false positives.

## Exit ticket
**Question:** Why would optimizing each test query using its true label create a challenging deployment assumption?

Record your answers in a new Markdown cell before moving to the next notebook. Many questions have more than one reasonable answer; explain your evidence.